### CELL 1: Import thư viện và Cấu hình

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, classification_report

# Import cấu hình và hàm tiện ích
from config import *
from data_utils import deep_clean_text
from model_utils import plot_dl_confusion_matrix, plot_prediction_confidence

# Đường dẫn file dữ liệu WELFake (Bạn hãy điều chỉnh cho khớp với thực tế)
WELFAKE_CSV_PATH = '../data/raw/WELFake_Dataset.csv' 

print("Đã load xong thư viện!")

### CELL 2: Tải Mô hình và Tokenizer (ISOT Pre-trained)

In [ ]:
print("======================================================")
print("KIỂM CHỨNG CHÉO MÔ HÌNH DEEP LEARNING TRÊN WELFAKE")
print("======================================================")

# 1. Tải bộ từ vựng Tokenizer
try:
    with open(TOKENIZER_PATH, 'rb') as f:
        tokenizer = pickle.load(f)
    print("Đã tải thành công Tokenizer (ISOT Vocabulary)!")
except FileNotFoundError:
    print(f"Không tìm thấy Tokenizer tại {TOKENIZER_PATH}.")

# 2. Tải Mô hình Bi-LSTM
try:
    bilstm_model = load_model(BILSTM_MODEL_PATH)
    print("Đã tải thành công mô hình Bi-LSTM!")
except FileNotFoundError:
    print(f"Không tìm thấy mô hình Bi-LSTM tại {BILSTM_MODEL_PATH}.")

# 3. Tải Mô hình 1D CNN (BỔ SUNG)
try:
    cnn_model = load_model(CNN_MODEL_PATH)
    print("Đã tải thành công mô hình 1D CNN!")
except FileNotFoundError:
    print(f"Không tìm thấy mô hình 1D CNN tại {CNN_MODEL_PATH}.")

### CELL 3: Tiền xử lý văn bản WELFake

In [ ]:
from data_utils import lemmatize_and_remove_stopwords

print("Đang tải bộ dữ liệu WELFake...")
welfake_df = pd.read_csv(WELFAKE_CSV_PATH)

# Xử lý bỏ Null ở cả 2 cột quan trọng trước khi lấy mẫu
welfake_df = welfake_df.dropna(subset=['title', 'text']).reset_index(drop=True)

# LẤY NGẪU NHIÊN 10.000 MẪU ĐỂ NHẸ MÁY VÀ TỐI ƯU TỐC ĐỘ CPU
print("Đang trích xuất ngẫu nhiên 10.000 bài báo từ WELFake...")
welfake_df = welfake_df.sample(n=10000, random_state=RANDOM_STATE).reset_index(drop=True)

print("Đang thực hiện gộp Title + Text và làm sạch thô (deep_clean_text)...")
# ĐỒNG BỘ TOÀN DIỆN: Gộp Title + Text y hệt như bên file Train
welfake_df['title_text'] = welfake_df['title'].astype(str) + " " + welfake_df['text'].astype(str)
welfake_df['cleaned_text'] = welfake_df['title_text'].apply(deep_clean_text).fillna('').astype(str).str.strip()

# Bỏ qua các bài viết rỗng sau khâu làm sạch thô
welfake_df = welfake_df[welfake_df['cleaned_text'] != ''].reset_index(drop=True)

print("Đang chuẩn hóa sâu (Lemmatization & Stopwords)...")
# Đưa từ về nguyên thể để Tokenizer đã lưu có thể tra cứu chính xác ngữ nghĩa
welfake_df['text_lemmatized'] = welfake_df['cleaned_text'].apply(lemmatize_and_remove_stopwords)

welfake_df = welfake_df[welfake_df['text_lemmatized'] != ''].reset_index(drop=True)

# Trích xuất nhãn và ma trận text đưa vào suy luận mô hình
y_welfake_true = welfake_df['label'].values
text_welfake = welfake_df['text_lemmatized'].values

print(f"Đã chuẩn bị xong {len(text_welfake):,} bài báo gộp từ tập WELFake để Test.")

### CELL 4: Tokenization & Padding (Vector hóa cho DL)

In [ ]:
print("Đang chuyển đổi văn bản thành chuỗi số (Tokenization)...")
X_welfake_seq = tokenizer.texts_to_sequences(text_welfake)

print("Đang đồng bộ chiều dài câu (Padding)...")
X_welfake_pad = pad_sequences(X_welfake_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print(f"Dữ liệu đã sẵn sàng! Shape ma trận đầu vào: {X_welfake_pad.shape}")

### CELL 5: Dự đoán và Đánh giá (Inference & Evaluation)

In [ ]:
print("Đang chạy mô hình dự đoán (Quá trình này có thể mất một lát)...")

# Suy luận xác suất
y_welfake_pred_probs_bilstm = bilstm_model.predict(X_welfake_pad)
y_welfake_pred_probs_cnn = cnn_model.predict(X_welfake_pad)

# Ép xác suất về nhãn 0 hoặc 1 (Ngưỡng 0.5)
y_welfake_pred_class_bilstm = (y_welfake_pred_probs_bilstm >= 0.5).astype(int).flatten()
y_welfake_pred_class_cnn = (y_welfake_pred_probs_cnn >= 0.5).astype(int).flatten()

# ĐÁNH GIÁ BI-LSTM
print("\n" + "="*60)
print("[KẾT QUẢ EXTERNAL VALIDATION TRÊN WELFAKE - BI-LSTM]")
print("="*60)
print(f"Accuracy: {accuracy_score(y_welfake_true, y_welfake_pred_class_bilstm):.4f}")
print("\nChi tiết (Classification Report):")
print(classification_report(y_welfake_true, y_welfake_pred_class_bilstm, target_names=['True News (0)', 'Fake News (1)']))

print("\nĐang xuất Ma trận nhầm lẫn (Confusion Matrix) cho Bi-LSTM...")
plot_dl_confusion_matrix(y_welfake_true, y_welfake_pred_class_bilstm, model_name="Bi-LSTM (Test on WELFake)")

print("\nĐang xuất Phân phối độ tự tin (Confidence) cho Bi-LSTM...")
plot_prediction_confidence(y_welfake_true, y_welfake_pred_probs_bilstm, model_name="Bi-LSTM (Test on WELFake)")


# ĐÁNH GIÁ 1D CNN (BỔ SUNG)
print("\n" + "="*60)
print("[KẾT QUẢ EXTERNAL VALIDATION TRÊN WELFAKE - 1D CNN]")
print("="*60)
print(f"Accuracy: {accuracy_score(y_welfake_true, y_welfake_pred_class_cnn):.4f}")
print("\nChi tiết (Classification Report):")
print(classification_report(y_welfake_true, y_welfake_pred_class_cnn, target_names=['True News (0)', 'Fake News (1)']))

print("\nĐang xuất Ma trận nhầm lẫn (Confusion Matrix) cho 1D CNN...")
plot_dl_confusion_matrix(y_welfake_true, y_welfake_pred_class_cnn, model_name="1D CNN (Test on WELFake)")

print("\nĐang xuất Phân phối độ tự tin (Confidence) cho 1D CNN...")
plot_prediction_confidence(y_welfake_true, y_welfake_pred_probs_cnn, model_name="1D CNN (Test on WELFake)")